### Calculator of expected credit loss car loan

##### This is a calculator according with the local regulation of Mexico CNBV

##### It can use just for one credit, like a calculator for one credit

##### These are the values of a car loan

In [46]:
# outstanding balance
ead_final = 1
# real payment 
pmt = 1 # payment scheduled in the credit amortization
pmt_rea1 = 0.50
pmt_rea2 = 0.50
pmt_rea3 = 1
pmt_rea4 = 1
# requisite amount
am_req1 = 1
am_req2 = 1
am_req3 = 1
am_req4 = 1

# total
tot_pmt_rea = 3
tot_am_req = 4 

an_int_rate = 61 # it must be integer value
rem_term = 900 # it must integer value (days)
v_age = 42
bkatr = 13 # how many consecutive months has the credit/account been in current
am_pay_cb = 3 # amount to pay to Credit Bureaus
inc = 1

##### These are fixed values, it can´t change

In [2]:
debt = 0.65
age = 54
# betas
b0 = -2.0471
b1 = 1.0837
b2 = -0.7863
b3 = 0.5473
b4 = 0.0587
b5 = -0.606
b6 = -0.1559
# loss given default values
lgd_b04 = 0.86
lgd_b45 = 0.91
lgd_b56 = 0.94
lgd_b67 = 0.95
lgd_b78 = 0.97
lgd_b89 = 0.98
lgd_b910 = 0.99
lgd_gt10 = 1.0

## Libraries

In [3]:

import numpy as np
import math as math


### Main parameters to obtain PD, LGD, EAD

#### Items for the parameters, these can have a different value according with the characteristic of the credit loan

In [47]:
pctj_paid1 = pmt_rea1 / am_req1
pctj_paid2 = pmt_rea2 / am_req2
pctj_paid3 = pmt_rea3 / am_req3
pctj_paid4 = pmt_rea4 / am_req4
v_debt = am_pay_cb / inc
mean4m_pctj_paid = (pctj_paid1 + pctj_paid2 + pctj_paid3+ pctj_paid4 ) * 0.25
bucket = math.ceil((tot_am_req - tot_pmt_rea) / pmt )
stage = np.where (bucket < 0, -1,np.where(bucket >= 0 and bucket <= 1, 1, np.where (\
        bucket > 1 and bucket <= 3, 2,np.where(bucket > 3, 3, -1))))
time_n = max(rem_term / 365.25,1)

#### Risk Levels

In [48]:
high = np.where(v_debt > debt and v_age <= age, 1, 0)

medium = np.where (v_debt <= debt and v_age <= age | v_debt > debt and v_age > age, 1, 0)

low = np.where (v_debt <= debt and v_age > age, 1, 0)

#### Caculate SP

In [49]:
lgd_final = np.where(bucket >= 0 and bucket <= 4, lgd_b04, np.where(bucket > 4 and bucket <= 5, lgd_b45, \
            np.where( bucket > 5 and bucket <= 6, lgd_b56, np.where( bucket > 6 and bucket <= 7, lgd_b67, \
            np.where ( bucket > 7 and bucket <= 8, lgd_b78, np.where (bucket > 8 and bucket <= 9, lgd_b89, \
            np.where ( bucket > 9 and bucket <= 10, lgd_b910, np.where (bucket > 10, lgd_gt10, -1 ))))))))

#### Calculate Z value

In [50]:
Z = - ( b0 + b1*mean4m_pctj_paid + b2*bucket + b3*high + b4*medium + b5*low + b6*bkatr )

#### Calculate PD

In [51]:
pdf_final = np.where (bucket > 3, 1, np.where (bucket <= 3, 1 / (1 + math.e**(Z)), -1))

#### Calculate ECL

In [52]:
ecl_trad = ead_final * lgd_final * pdf_final
item_a = (ead_final * lgd_final * pdf_final)/(1 + an_int_rate/100) * ((1 - (1 - pdf_final) ** (time_n)) / pdf_final)
item_b = (lgd_final * pdf_final * pmt * 12) / ((1 + an_int_rate/100) * an_int_rate/100) *\
         ((1- (1 - pdf_final)**(time_n))/pdf_final)
item_c = (lgd_final * pdf_final * pmt* 12) / ((pdf_final + an_int_rate/100) * an_int_rate/100 ) *\
         (1 - ((1 - pdf_final) / (1 + an_int_rate/100)) ** (time_n))
ecl_ltime =  item_a - item_b + item_c
ecl_stage2 = max (ecl_trad, item_a - item_b + item_c)
ecl_final = np.where (stage == 1 or stage == 3,ecl_trad, ecl_stage2 ) 

In [ ]:
print(np.round(ecl_final,decimals = 4),np.round(ecl_ltime,decimals = 2 ), pdf_final ,lgd_final,stage, bucket, sep = "|")

0.0252|-0.15|0.029311519434129577|0.86|1|1
